TLDR : 

1. load the packaged low-z projected-truth dataset and config;
2. inspect redshift, photometric errors, photometry, and DSPS input truth columns;
3. define the neural network and optional compact training loop;
4. load the active weights, from the package or from notebook retraining;
5. run inference on a configurable number of rows;
6. plot the global residual distribution and residuals by band.


In [ ]:
from __future__ import annotations

import os
import sys
from copy import deepcopy
from pathlib import Path

# Notebook analysis stack.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# IPython display is optional so the notebook can still be smoke-tested as a script.
try:
    from IPython.display import display
except ImportError:
    display = print


# Package discovery
# -----------------
# The zip is portable: it works when the notebook is opened from the package
# root, from package/notebooks, or from the source checkout package copy.
PACKAGE_MARKER = Path("data/nokl_trainval20k_projected_truth.parquet")
REPO = Path(os.environ.get("DSPS_REPO", "/home/maxime/src/DSPS")).expanduser().resolve()
DEFAULT_PACKAGE_DIR = REPO / "outputs/deliverables/diffsky_nokl_lowz_baseline"

candidate_dirs = []
if os.environ.get("DIFFSKY_PACKAGE_DIR"):
    candidate_dirs.append(Path(os.environ["DIFFSKY_PACKAGE_DIR"]).expanduser().resolve())

cwd = Path.cwd().resolve()
candidate_dirs.extend([cwd, *cwd.parents])
candidate_dirs.append(DEFAULT_PACKAGE_DIR)

PACKAGE_DIR = None
for candidate in dict.fromkeys(candidate_dirs):
    if (candidate / PACKAGE_MARKER).exists():
        PACKAGE_DIR = candidate
        break

if PACKAGE_DIR is None:
    searched = "\n".join(str(path) for path in candidate_dirs)
    raise FileNotFoundError(
        "Could not find the packaged Diffsky data file. "
        "Run the notebook from the package root, or set DIFFSKY_PACKAGE_DIR.\n"
        f"Searched:\n{searched}"
    )

# Config paths are package-relative, so use the package root as the working dir.
os.chdir(PACKAGE_DIR)

# Add the local checkout only when it is present. Otherwise euclid-dsps must be
# installed in the active Python environment.
if str(REPO) not in sys.path and (REPO / "euclid_dsps").exists():
    sys.path.insert(0, str(REPO))


# Package inputs
# --------------
CONFIG_PATH = PACKAGE_DIR / "configs/diffsky_nokl_trainval20k.yaml"
TRAINVAL_DATASET_PATH = PACKAGE_DIR / "data/nokl_trainval20k_projected_truth.parquet"
FULL_DATASET_PATH = PACKAGE_DIR / "data/lowz_78651_projected_truth.parquet"
PROJECTED_TRUTH_METADATA_PATH = PACKAGE_DIR / "data/projected_truth_metadata.csv"
WEIGHT_PATH = PACKAGE_DIR / "weights/best.eqx"
FEATURE_STATS_PATH = PACKAGE_DIR / "weights/feature_stats.json"

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})
pd.set_option("display.max_columns", 160)


# Quick file check
# ----------------
print(f"PACKAGE_DIR = {PACKAGE_DIR}")
for path in [CONFIG_PATH, TRAINVAL_DATASET_PATH, FULL_DATASET_PATH, WEIGHT_PATH, FEATURE_STATS_PATH]:
    print(f"{path.relative_to(PACKAGE_DIR)} exists={path.exists()}")


In [ ]:
# Optional euclid-dsps install
# ----------------------------
# The notebook needs the euclid_dsps Python package. If it is already installed,
# this cell does nothing. If it is missing, the default behavior is to clone the
# public branch used for this notebook and install it in editable mode.
import importlib.util
import subprocess

AUTO_INSTALL_EUCLID_DSPS = True
EUCLID_DSPS_REPO_URL = "https://github.com/CosmoStat/euclid-dsps-shine.git"
EUCLID_DSPS_BRANCH = "feature/diffsky-likelihood-sanity-plan"
EUCLID_DSPS_LOCAL_DIR = PACKAGE_DIR / "_deps/euclid-dsps-shine"

if importlib.util.find_spec("euclid_dsps") is None:
    if not AUTO_INSTALL_EUCLID_DSPS:
        raise ModuleNotFoundError(
            "euclid_dsps is not installed. Set AUTO_INSTALL_EUCLID_DSPS=True "
            "in this cell, or install the repository manually."
        )

    EUCLID_DSPS_LOCAL_DIR.parent.mkdir(parents=True, exist_ok=True)

    if not (EUCLID_DSPS_LOCAL_DIR / ".git").exists():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                EUCLID_DSPS_BRANCH,
                "--single-branch",
                EUCLID_DSPS_REPO_URL,
                str(EUCLID_DSPS_LOCAL_DIR),
            ]
        )
    else:
        subprocess.check_call(["git", "fetch", "origin", EUCLID_DSPS_BRANCH], cwd=EUCLID_DSPS_LOCAL_DIR)
        subprocess.check_call(["git", "checkout", EUCLID_DSPS_BRANCH], cwd=EUCLID_DSPS_LOCAL_DIR)
        subprocess.check_call(["git", "pull", "--ff-only", "origin", EUCLID_DSPS_BRANCH], cwd=EUCLID_DSPS_LOCAL_DIR)

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(EUCLID_DSPS_LOCAL_DIR)])
    importlib.invalidate_caches()

if importlib.util.find_spec("euclid_dsps") is None:
    raise ModuleNotFoundError("euclid_dsps is still not importable after installation.")

print("euclid_dsps is importable")


In [ ]:
# Load config and packaged data
# -----------------------------
# Keep this cell boring: load the YAML, load the parquet tables, and print only
# the contract needed to read the rest of the notebook.
from euclid_dsps.config import load_config

config = load_config(CONFIG_PATH)

bands = config["bands"]
band_names = [band["name"] for band in bands]
flux_cols = [band["column"] for band in bands]
err_cols = [band["error_column"] for band in bands]

trainval = pd.read_parquet(TRAINVAL_DATASET_PATH)
full_lowz = pd.read_parquet(FULL_DATASET_PATH)
truth_metadata = pd.read_csv(PROJECTED_TRUTH_METADATA_PATH)

amortized = config["amortized"]
encoder_cfg = amortized["encoder"]
training_cfg = amortized["training"]
likelihood_cfg = amortized["likelihood"]
calibration_cfg = config.get("calibration", {}) or {}

split_counts = trainval["nokl_split"].value_counts().to_dict()
feature_dim = encoder_cfg["input_dim"]
latent_schema = amortized["latent"]["schema"]

important_config = pd.DataFrame(
    [
        ("config", "file", CONFIG_PATH.relative_to(PACKAGE_DIR)),
        ("data", "train/validation parquet", TRAINVAL_DATASET_PATH.relative_to(PACKAGE_DIR)),
        ("data", "full low-z parquet", FULL_DATASET_PATH.relative_to(PACKAGE_DIR)),
        ("data", "full low-z rows", f"{len(full_lowz):,}"),
        ("data", "train/validation rows", f"{len(trainval):,}"),
        ("data", "split counts", split_counts),
        ("photometry", "bands", f"{len(band_names)} bands: {', '.join(band_names)}"),
        ("features", "encoder input", f"{feature_dim} = {len(band_names)} fluxes + {len(band_names)} errors"),
        ("network", "MLP", f"{encoder_cfg['hidden_sizes']} + {encoder_cfg['activation'].upper()}"),
        ("network", "latent dimension", encoder_cfg["latent_dim"]),
        ("network", "latent schema", latent_schema),
        ("objective", "likelihood", likelihood_cfg.get("type")),
        ("objective", "Student-t dof", likelihood_cfg.get("student_t_dof")),
        ("objective", "error floor frac", likelihood_cfg.get("error_floor_frac")),
        ("objective", "KL weight max", training_cfg.get("kl_weight_max", 0.0)),
        ("training", "epochs", training_cfg.get("epochs")),
        ("training", "batch size", training_cfg.get("batch_size")),
        ("calibration", "global SED scale", calibration_cfg.get("global_sed_scale", {}).get("mode")),
        ("calibration", "per-band zero points", calibration_cfg.get("per_band_zero_points", {}).get("mode")),
    ],
    columns=["section", "item", "value"],
)

display(important_config)

print(f"config loaded: {CONFIG_PATH.relative_to(PACKAGE_DIR)}")
print(f"dataset rows: train/validation={len(trainval):,}, full_lowz={len(full_lowz):,}")
print(f"bands ({len(band_names)}): {', '.join(band_names)}")
print(f"truth metadata rows: {len(truth_metadata)}")


In [ ]:
# Redshift and split EDA
# ----------------------
# The 20k subset is the exact train/validation row set used by the reference NN  checkpoint
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

axes[0].hist(full_lowz["redshift_true"], bins=70, histtype="step", lw=1.5)
axes[0].set_title("Full low-z catalog")
axes[0].set_xlabel("redshift_true")

for split_name, frame in trainval.groupby("nokl_split"):
    axes[1].hist(frame["redshift_true"], bins=45, histtype="step", lw=1.5, label=split_name)
axes[1].set_title("20k train/validation subset")
axes[1].set_xlabel("redshift_true")
axes[1].legend()

axes[2].hist(trainval["redshift_true"], bins=45, histtype="stepfilled", alpha=0.35)
axes[2].set_title("Subset redshift distribution")
axes[2].set_xlabel("redshift_true")

fig.tight_layout()


In [ ]:
# Photometry and error-model EDA
# ------------------------------
# Fluxes and errors are already in fnu_cgs units. The model input is not raw
# magnitudes: it is normalized flux/error features computed from these columns.
flux = trainval[flux_cols].to_numpy(float)
err = trainval[err_cols].to_numpy(float)

frac_err = np.divide(err, np.abs(flux), out=np.full_like(err, np.nan), where=np.abs(flux) > 0)

photometry_summary = pd.DataFrame(
    {
        "band": band_names,
        "flux_median": np.nanmedian(flux, axis=0),
        "flux_q16": np.nanpercentile(flux, 16, axis=0),
        "flux_q84": np.nanpercentile(flux, 84, axis=0),
        "fluxerr_median": np.nanmedian(err, axis=0),
        "frac_err_median": np.nanmedian(frac_err, axis=0),
        "frac_err_q84": np.nanpercentile(frac_err, 84, axis=0),
    }
)
display(photometry_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].boxplot([flux[:, index] for index in range(len(band_names))], tick_labels=band_names, showfliers=False)
axes[0].set_yscale("symlog", linthresh=1.0e-32)
axes[0].set_title("Observed flux by band")
axes[0].tick_params(axis="x", rotation=45)

axes[1].boxplot([frac_err[:, index] for index in range(len(band_names))], tick_labels=band_names, showfliers=False)
axes[1].set_yscale("log")
axes[1].set_title("Fractional error by band")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()


In [ ]:
# DSPS truth columns
# ------------------------
# Some DSPS latent inputs have catalog/projected truth columns. Simple EDA distribution plot cell
DSPS_INPUT_TRUTH_COLUMNS = [
    "z_obs",
    "log10_stellar_mass",
    "dlog10_sfr_1",
    "dlog10_sfr_2",
    "dlog10_sfr_3",
    "dlog10_sfr_4",
    "dlog10_sfr_5",
    "dlog10_sfr_6",
    "log10_stellar_metallicity",
    "tau2",
    "dust_index_n",
    "tau1_over_tau2",
]

truth_source = truth_metadata.set_index("parameter")["source_kind"].to_dict()
truth_table = pd.DataFrame(
    {
        "parameter": DSPS_INPUT_TRUTH_COLUMNS,
        "source_kind": [truth_source.get(name, "") for name in DSPS_INPUT_TRUTH_COLUMNS],
        "finite_fraction": [
            np.isfinite(pd.to_numeric(trainval[name], errors="coerce")).mean()
            for name in DSPS_INPUT_TRUTH_COLUMNS
        ],
    }
)

# Metallicity note:
# DSPS needs stellar metallicity to choose/interpolate SSP spectra. The catalog
# used here does not provide a reliable stellar-metallicity truth column, so the
# NN encoder still outputs log10_stellar_metallicity, but it is learned only
# through the photometric reconstruction likelihood. 
metallicity_note = pd.DataFrame(
    [
        {
            "parameter": "log10_stellar_metallicity",
            "used_by_dsps": True,
            "truth_available_in_this_dataset": False,
            "how_training_handles_it": "encoder predicts it because DSPS needs it; loss is photometric reconstruction only",
            "interpretation": "inferred nuisance parameter, not catalog truth",
        }
    ]
)

display(truth_table)
display(metallicity_note)

fig, axes = plt.subplots(3, 4, figsize=(13, 8))
for ax, column in zip(axes.ravel(), DSPS_INPUT_TRUTH_COLUMNS, strict=True):
    values = pd.to_numeric(trainval[column], errors="coerce").to_numpy(float)
    values = values[np.isfinite(values)]
    if values.size:
        ax.hist(values, bins=60, histtype="step")
    else:
        ax.text(0.5, 0.5, "no truth", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(column, fontsize=9)

fig.tight_layout()


In [ ]:
# Neural-network definition
# -------------------------
import jax
import jax.numpy as jnp

# read_feature_stats reloads the flux/error normalization saved next to the checkpoint.
from euclid_dsps.amortized.features import read_feature_stats

# latent_spec_from_config defines the DSPS latent parameter order/bounds.
# x_to_theta maps NN latent coordinates back to physical DSPS values.
from euclid_dsps.amortized.latent import latent_spec_from_config, x_to_theta

# architecture_summary prints the configured MLP/latent sizes.
# build_amortized_model instantiates the Equinox model matching the checkpoint.
from euclid_dsps.amortized.train import architecture_summary, build_amortized_model

SEED = int(config["amortized"]["training"].get("seed", 42))
latent_spec = latent_spec_from_config(config)
feature_stats = read_feature_stats(FEATURE_STATS_PATH)
model_template = build_amortized_model(config, jax.random.PRNGKey(SEED))

encoder_flow = pd.DataFrame(
    [
        ("input", "normalized flux/error features", config["amortized"]["encoder"]["input_dim"]),
        ("hidden 1", "Linear + GELU", 192),
        ("hidden 2", "Linear + GELU", 192),
        ("hidden 3", "Linear + GELU", 192),
        ("mean head", "Linear", len(latent_spec.names)),
        ("log_std head", "Linear + clip", len(latent_spec.names)),
    ],
    columns=["stage", "operation", "output_dim"],
)

print("Neural network / latent contract")
print(f"input_dim = {config['amortized']['encoder']['input_dim']} = 14 flux features + 14 error features")
print(f"latent_dim = {len(latent_spec.names)}")
print(f"latent names = {latent_spec.names}")
print("Equinox model repr follows; remember GELU is stored as activation_name, not as a module.")

display(encoder_flow)
display(pd.DataFrame([architecture_summary(config)]).T.rename(columns={0: "value"}))
print(model_template.encoder)


In [ ]:
# (Optional) retraining cell
# ------------------------
# RUN_TRAINING stays False for the delivered notebook. Set it to True to retrail
RUN_TRAINING = False
RETRAIN_OUT = PACKAGE_DIR / "runs/retrain_nokl"

from euclid_dsps.amortized.config import amortized_config, require_amortized_dependencies
from euclid_dsps.amortized.data import iter_photometry_batches_from_arrays, load_photometry_arrays_from_config
from euclid_dsps.amortized.features import compute_feature_stats, write_feature_stats
from euclid_dsps.amortized.train import (
    JitLatentSpec,
    LossBatch,
    _StaticArg,
    _effective_jax_batch_size,
    _kl_weight,
    _loss_and_grads_jit,
    _metrics_record,
    _objective_config_for_epoch,
    build_amortized_model,
    make_optimizer,
    save_checkpoint,
    tree_all_finite,
    zero_band_calibration_grads,
    zero_prior_grads,
    zero_sed_scale_grads,
)
from euclid_dsps.calibration import global_sed_scale_config, per_band_flux_calibration_config
from euclid_dsps.filters import load_filters
from euclid_dsps.model import dynamic_model_args, load_context

# Equinox is used only for filtered optimizer state/update handling.
eqx, _ = require_amortized_dependencies()


def _finite_mean(values):
    """Mean over finite scalar metric values."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(values.mean()) if values.size else float("nan")


def simple_train_amortized_fs2(
    out_dir: Path = RETRAIN_OUT,
    epochs: int = 30,
    batch_size: int = 128,
    seed: int = 42,
) -> Path:
    """Simple training loop

    feature normalization, encoder+DSPS loss, Student-t likelihood,
    global SED scale, per-band zero points, AdamW, gradient clipping,
    validation best checkpoint selection.

    """
    out_dir = Path(out_dir)
    ckpt_dir = out_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # Force the delivered no-KL objective even if somebody edits the YAML later.
    train_config = deepcopy(config)
    train_config["amortized"]["training"]["kl_weight_max"] = 0.0
    train_config["amortized"]["training"]["kl_annealing_epochs"] = 1
    cfg = amortized_config(train_config)

    # Reuse the exact train/validation row positions from the reference run.
    train_indices = np.load(PACKAGE_DIR / "configs/train_indices_subset_positions.npy")
    validation_indices = np.load(PACKAGE_DIR / "configs/validation_indices_subset_positions.npy")

    catalog_batch_size = int(cfg["data"].get("catalog_batch_size", max(int(batch_size), 10_000)))
    jax_batch_size = _effective_jax_batch_size(cfg["training"], int(batch_size))

    train_arrays = load_photometry_arrays_from_config(
        train_config,
        batch_size=catalog_batch_size,
        row_indices=train_indices,
    )
    validation_arrays = load_photometry_arrays_from_config(
        train_config,
        batch_size=catalog_batch_size,
        row_indices=validation_indices,
    )

    # The encoder never sees raw fluxes directly. It sees normalized features
    # computed from the train split only, then reused for validation/inference.
    feature_stats = compute_feature_stats(
        train_arrays.flux,
        train_arrays.flux_err,
        train_arrays.mask,
        band_names=train_arrays.band_names,
        flux_transform=str(cfg["features"].get("flux_transform", "asinh")),
    )
    write_feature_stats(out_dir / "feature_stats.json", feature_stats)

    # DSPS forward-model context: filters, SSP basis, and fixed runtime args.
    filters = load_filters(train_config["bands"])
    context = load_context(
        train_config["ssp_path"],
        filters,
        n_sfh_bins=int(train_config["model"].get("n_sfh_bins", 96)),
        cosmos_config=train_config.get("cosmos_sed"),
        nebular_emission=train_config.get("nebular_emission", "ssp_flux"),
        model_config=train_config.get("model"),
    )
    model_args = dynamic_model_args(context)
    jit_context = _StaticArg(context)

    # Latent bounds/order must match the checkpoint and the DSPS decoder.
    latent_spec = latent_spec_from_config(train_config)
    jit_latent_spec = JitLatentSpec(
        names=latent_spec.names,
        lower=latent_spec.lower,
        upper=latent_spec.upper,
        raw_center=latent_spec.raw_center,
        raw_scale=latent_spec.raw_scale,
        normalization=latent_spec.normalization,
    )

    key = jax.random.PRNGKey(int(seed))
    key, model_key = jax.random.split(key)
    model = build_amortized_model(train_config, model_key)

    optimizer = make_optimizer(train_config)
    opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

    # Calibration parameters are trainable in the reference no-KL run.
    calibration_config = {"calibration": train_config.get("calibration", {}) or {}}
    sed_scale_cfg = global_sed_scale_config(calibration_config)
    band_calibration_cfg = per_band_flux_calibration_config(calibration_config)
    train_sed_scale = bool(sed_scale_cfg.enabled and sed_scale_cfg.trainable)
    train_band_calibration = bool(
        band_calibration_cfg.enabled
        and band_calibration_cfg.trainable
        and model.band_calibration is not None
    )

    best_min_epoch = int(cfg["training"].get("best_checkpoint_min_epoch", 2))
    best_validation_nll = np.inf
    rows = []
    train_rng = np.random.default_rng(int(seed) + 10_000)

    for epoch in range(1, int(epochs) + 1):
        objective_config = _objective_config_for_epoch(cfg, epoch)
        kl_weight = _kl_weight(
            epoch,
            int(cfg["training"].get("kl_annealing_epochs", 1)),
            max_weight=0.0,
        )

        # Shuffle train rows every epoch, matching the production run behavior.
        train_order = np.arange(len(train_arrays.object_id))
        if bool(cfg["data"].get("epoch_shuffle", True)):
            train_rng.shuffle(train_order)

        epoch_train_rows = []
        for batch_index, batch in enumerate(
            iter_photometry_batches_from_arrays(
                train_arrays,
                batch_size=jax_batch_size,
                feature_stats=feature_stats,
                order=train_order,
            )
        ):
            key, step_key = jax.random.split(key)
            loss_batch = LossBatch(batch.flux, batch.flux_err, batch.mask, batch.features)

            (loss, metrics), grads = _loss_and_grads_jit(
                model,
                loss_batch,
                jit_latent_spec,
                jit_context,
                model_args,
                latent_spec.names,
                step_key,
                1,
                kl_weight,
                cfg["likelihood"],
                calibration_config,
                objective_config,
            )

            # no-KL baseline: train encoder/calibration, keep the RealNVP prior frozen.
            grads = zero_prior_grads(grads)
            if not train_sed_scale:
                grads = zero_sed_scale_grads(grads)
            if not train_band_calibration:
                grads = zero_band_calibration_grads(grads)

            loss_finite = bool(np.isfinite(float(np.asarray(jax.device_get(loss)))))
            grads_finite = tree_all_finite(grads)
            update_applied = bool(loss_finite and grads_finite)

            if update_applied:
                updates, opt_state = optimizer.update(
                    grads,
                    opt_state,
                    eqx.filter(model, eqx.is_inexact_array),
                )
                model = eqx.apply_updates(model, updates)

            record = _metrics_record(metrics)
            record.update(
                {
                    "split": "train",
                    "epoch": epoch,
                    "batch": batch_index,
                    "update_applied": float(update_applied),
                }
            )
            rows.append(record)
            epoch_train_rows.append(record)

        # Validation uses the same loss but applies no optimizer update.
        validation_rows = []
        for batch_index, batch in enumerate(
            iter_photometry_batches_from_arrays(
                validation_arrays,
                batch_size=jax_batch_size,
                feature_stats=feature_stats,
            )
        ):
            key, step_key = jax.random.split(key)
            loss_batch = LossBatch(batch.flux, batch.flux_err, batch.mask, batch.features)

            (_, metrics), _ = _loss_and_grads_jit(
                model,
                loss_batch,
                jit_latent_spec,
                jit_context,
                model_args,
                latent_spec.names,
                step_key,
                1,
                kl_weight,
                cfg["likelihood"],
                calibration_config,
                objective_config,
            )

            record = _metrics_record(metrics)
            record.update(
                {
                    "split": "validation",
                    "epoch": epoch,
                    "batch": batch_index,
                    "update_applied": 0.0,
                }
            )
            rows.append(record)
            validation_rows.append(record)

        train_nll = _finite_mean([row["negative_loglike"] for row in epoch_train_rows])
        validation_nll = _finite_mean([row["negative_loglike"] for row in validation_rows])

        save_checkpoint(
            ckpt_dir / "last.eqx",
            model,
            config=train_config,
            latent_spec=latent_spec,
            feature_stats=feature_stats,
            epoch=epoch,
            metric=train_nll,
            metric_name="train_negative_loglike",
        )

        if epoch >= best_min_epoch and np.isfinite(validation_nll) and validation_nll < best_validation_nll:
            best_validation_nll = validation_nll
            save_checkpoint(
                ckpt_dir / "best.eqx",
                model,
                config=train_config,
                latent_spec=latent_spec,
                feature_stats=feature_stats,
                epoch=epoch,
                metric=best_validation_nll,
                metric_name="validation_negative_loglike",
            )

        pd.DataFrame(rows).to_csv(out_dir / "training_log.csv", index=False)
        print(
            f"epoch {epoch:03d}: "
            f"train_nll={train_nll:.6g} "
            f"validation_nll={validation_nll:.6g} "
            f"best_validation_nll={best_validation_nll:.6g}"
        )

    if not (ckpt_dir / "best.eqx").exists():
        save_checkpoint(
            ckpt_dir / "best.eqx",
            model,
            config=train_config,
            latent_spec=latent_spec,
            feature_stats=feature_stats,
            epoch=int(epochs),
            metric=train_nll,
            metric_name="train_negative_loglike",
        )

    return ckpt_dir / "best.eqx"


if RUN_TRAINING:
    simple_train_amortized_fs2()
else:
    print(
        "RUN_TRAINING=False: using provided weights. "
        "Set True to retrain and write runs/retrain_nokl/checkpoints/best.eqx"
    )


In [ ]:
# Load best weights
# -----------------
# By default this loads the provided checkpoint. If RUN_TRAINING=True above,
# it instead loads the best checkpoint produced by the notebook retraining cell.
from euclid_dsps.amortized.train import load_checkpoint

ACTIVE_WEIGHT_PATH = RETRAIN_OUT / "checkpoints/best.eqx" if RUN_TRAINING else WEIGHT_PATH
ACTIVE_FEATURE_STATS_PATH = RETRAIN_OUT / "feature_stats.json" if RUN_TRAINING else FEATURE_STATS_PATH

if not ACTIVE_WEIGHT_PATH.exists():
    raise FileNotFoundError(ACTIVE_WEIGHT_PATH)
if not ACTIVE_FEATURE_STATS_PATH.exists():
    raise FileNotFoundError(ACTIVE_FEATURE_STATS_PATH)

model = load_checkpoint(ACTIVE_WEIGHT_PATH, config)
feature_stats = read_feature_stats(ACTIVE_FEATURE_STATS_PATH)

print(f"loaded weights: {ACTIVE_WEIGHT_PATH.relative_to(PACKAGE_DIR)}")
print(f"loaded feature stats: {ACTIVE_FEATURE_STATS_PATH.relative_to(PACKAGE_DIR)}")


In [ ]:
# Inference with loaded weights
# -----------------------------
from euclid_dsps.amortized.decoder import model_flux_from_x
from euclid_dsps.amortized.features import make_encoder_features
from euclid_dsps.amortized.diagnostics import posterior_predictive_residual_summary_frame
from euclid_dsps.calibration import (
    apply_global_sed_scale_to_flux,
    apply_per_band_flux_calibration_to_flux,
    global_sed_scale_config,
    per_band_flux_calibration_config,
)
from euclid_dsps.filters import load_filters
from euclid_dsps.model import dynamic_model_args, load_context
from euclid_dsps.observation_arrays import photometry_arrays_from_dataframe

# Keep this small by default 
# when you want the full 20k subset residual diagnostics from the active weights.
N_INFER = 1000
infer_frame = trainval.head(min(N_INFER, len(trainval))).copy()

arrays = photometry_arrays_from_dataframe(
    infer_frame,
    config["bands"],
    object_id_column="object_id",
)
features = make_encoder_features(
    jnp.asarray(arrays.flux),
    jnp.asarray(arrays.flux_err),
    feature_stats,
)

# We use the encoder mean for a deterministic diagnostic in the notebook. The full posterior predictive actually sample from the latent
mean_x, log_std_x = model.encoder(features)
theta = pd.DataFrame(
    np.asarray(jax.device_get(x_to_theta(mean_x, latent_spec))),
    columns=latent_spec.names,
)
theta.insert(0, "object_id", infer_frame["object_id"].to_numpy())
theta.insert(0, "row_index", infer_frame["row_index"].to_numpy())

filters = load_filters(config["bands"])
context = load_context(
    config["ssp_path"],
    filters,
    n_sfh_bins=int(config["model"].get("n_sfh_bins", 80)),
    cosmos_config=config.get("cosmos_sed"),
    nebular_emission=config.get("nebular_emission", "ssp_flux"),
    model_config=config.get("model"),
)
model_args = dynamic_model_args(context)
model_flux = model_flux_from_x(mean_x, latent_spec, context, model_args, latent_spec.names)

# Apply the same learned calibration parameters used by the trained model.
scale_cfg = global_sed_scale_config({"calibration": config.get("calibration", {}) or {}})
if scale_cfg.enabled:
    model_flux = apply_global_sed_scale_to_flux(model_flux, model.sed_scale.log_alpha_sed)

band_cfg = per_band_flux_calibration_config({"calibration": config.get("calibration", {}) or {}})
if band_cfg.enabled and model.band_calibration is not None:
    model_flux = apply_per_band_flux_calibration_to_flux(
        model_flux,
        model.band_calibration.log_alpha_band,
    )

model_flux = np.asarray(jax.device_get(model_flux), dtype=float)[None, :, :]
inference_residual_summary = posterior_predictive_residual_summary_frame(
    arrays.object_id,
    arrays.flux,
    arrays.flux_err,
    arrays.mask,
    model_flux,
    arrays.band_names,
    row_index=infer_frame["row_index"].to_numpy(),
    likelihood_config=config["amortized"]["likelihood"],
)

n_objects = len(infer_frame)
n_bands = len(arrays.band_names)
expected_summary_rows = n_objects * n_bands

# The residual summary is long-form: one row per object and per band.
# Example: N_INFER=4 and 14 bands gives 4 * 14 = 56 rows.
print(f"inference objects = {n_objects}")
print(f"bands = {n_bands}: {list(arrays.band_names)}")
print(f"residual summary rows = {len(inference_residual_summary)} = {n_objects} objects x {n_bands} bands")
assert len(inference_residual_summary) == expected_summary_rows
print(f"active weights = {ACTIVE_WEIGHT_PATH.relative_to(PACKAGE_DIR)}")

display(theta.head())
display(inference_residual_summary.head())


In [ ]:
# Global residual diagnostic
# --------------------------
# Plot the likelihood-normalized residuals from the active inference above.
def _residual_column(summary: pd.DataFrame) -> str:
    return "residual_sigma_median" if "residual_sigma_median" in summary else "chi_likelihood_median"


def plot_normalized_residual_hist(
    summary: pd.DataFrame,
    out_path: Path | None = None,
    *,
    title: str = "Likelihood-normalized inference residuals",
):
    residual_column = _residual_column(summary)
    values = pd.to_numeric(summary[residual_column], errors="coerce").to_numpy(float)
    values = values[np.isfinite(values)]

    fig, ax = plt.subplots(figsize=(7, 4))
    if values.size:
        ax.hist(values, bins=60, density=True, alpha=0.65, label="encoder mean")
        x = np.linspace(
            min(-6.0, float(np.nanpercentile(values, 1.0))),
            max(6.0, float(np.nanpercentile(values, 99.0))),
            400,
        )
        ax.plot(x, np.exp(-0.5 * x**2) / np.sqrt(2.0 * np.pi), color="black", lw=1.2, label="N(0,1)")

    ax.axvline(-3.0, color="tab:red", lw=1.0, ls="--")
    ax.axvline(3.0, color="tab:red", lw=1.0, ls="--")
    ax.axvline(0.0, color="black", lw=1.0, alpha=0.5)
    ax.set_xlabel("(flux_in - flux_out) / sigma_eff")
    ax.set_ylabel("density")
    ax.set_title(title)
    ax.legend(loc="best")
    fig.tight_layout()

    if out_path is not None:
        fig.savefig(out_path, dpi=150)
    return fig


residual_column = _residual_column(inference_residual_summary)
print(f"residual summary rows = {len(inference_residual_summary)}")
display(
    pd.to_numeric(inference_residual_summary[residual_column], errors="coerce")
    .describe(percentiles=[0.01, 0.16, 0.5, 0.84, 0.99])
    .to_frame(residual_column)
)

plot_normalized_residual_hist(
    inference_residual_summary,
    title=f"Loaded-weight inference residuals (N={len(infer_frame)})",
)


In [ ]:
# Residual diagnostic by band
# ---------------------------
# Same residual definition as above, grouped by photometric band.
def plot_residuals_by_band(
    summary: pd.DataFrame,
    out_path: Path | None = None,
    *,
    title: str = "Likelihood-normalized inference residuals by band",
):
    residual_column = _residual_column(summary)
    bands = list(dict.fromkeys(summary["band"].astype(str)))
    data = [
        pd.to_numeric(
            summary.loc[summary["band"].astype(str) == band, residual_column],
            errors="coerce",
        ).to_numpy(float)
        for band in bands
    ]

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.axhline(0.0, color="black", lw=1.0, alpha=0.5)
    ax.axhline(-3.0, color="tab:red", lw=1.0, alpha=0.6, ls="--")
    ax.axhline(3.0, color="tab:red", lw=1.0, alpha=0.6, ls="--")
    ax.boxplot(data, tick_labels=bands, showfliers=False)
    ax.set_ylabel("(flux_in - flux_out) / sigma_eff")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()

    if out_path is not None:
        fig.savefig(out_path, dpi=150)
    return fig


plot_residuals_by_band(
    inference_residual_summary,
    title=f"Loaded-weight inference residuals by band (N={len(infer_frame)})",
)
